In [ ]:
# =============================================================================
# HYDROSOIL-AI
# Prediction of soil saturated hydraulic conductivity (Ksat) from the Soil Water
# Infiltration Global (SWIG) database, with an evaluation design that respects the
# hierarchical structure of the compilation.
#
# INPUT   a SWIG database workbook (.xlsx) containing a metadata sheet and a
#         locations sheet
# OUTPUT  results_tables.xlsx, eight figures, the analysis table, a serialised
#         screening model and a run-metadata file
#
# -----------------------------------------------------------------------------
# WHY THE EVALUATION DESIGN MATTERS
#
# SWIG is a compilation of measurements contributed by many independent research
# groups. Records arrive in blocks: each block is one study, carried out at one or
# a few sites, with one instrument, one operator team and one measurement
# protocol. Records inside a block therefore share much more than their soil
# physics.
#
# If such data are split into training and test partitions at random, records from
# the same block land on both sides. A flexible model can then recognise the block
# from its covariate signature and reproduce that block's mean response. The
# resulting test score measures interpolation inside known studies, not prediction
# for new ones.
#
# This script quantifies that difference directly: it evaluates the same pipeline
# under a random row-wise split and under splits that keep every source dataset,
# country or geographic block entirely on one side of the partition.
#
# -----------------------------------------------------------------------------
# PIPELINE
#
#   PHASE  1  Locate and read the database, tolerant of file, sheet and column
#             naming differences
#   PHASE  2  Define the analysis sample, report every exclusion, and remove
#             exactly duplicated records
#   PHASE  3  Construct the physics-guided features, with explicit equations
#   PHASE  4  Set up the cross-validation machinery, models and metrics
#   PHASE  5  Experiment 1: compare four validation protocols; decompose the
#             result into level bias and within-study ranking skill
#   PHASE  6  Experiment 2: benchmark fifteen algorithms under two protocols
#   PHASE  7  Experiment 3: feature-set ablation against non-learned baselines
#   PHASE  8  Experiment 4: leave-one-instrument-out, leave-one-country-out and
#             variance components
#   PHASE  9  Experiment 5: out-of-fold permutation importance and SHAP
#   PHASE 10  Build and evaluate the deployable four-input model, with conformal
#             prediction intervals and an applicability domain
#   PHASE 11  Produce all figures
#   PHASE 12  Export every table, the analysis dataset and the run metadata
#
# -----------------------------------------------------------------------------
# METHODOLOGICAL RULES OBSERVED THROUGHOUT
#
#   * Imputation and scaling are fitted on the training partition of each outer
#     fold and applied to the held-out partition. No preprocessing sees the
#     evaluation data.
#   * Hyper-parameters are selected in a group-aware inner loop nested inside each
#     outer fold, so the reported score carries no model-selection optimism.
#   * Every stochastic component is seeded, including the estimator used inside
#     the imputer, so the run is reproducible.
#   * Feature importance is computed on held-out folds. Training-set attributions
#     are reported separately and only for contrast.
#   * The deployable model is evaluated with exactly the inputs its interface
#     requests, and is distributed with a prediction interval and an
#     applicability-domain check.
#   * Records that are exact duplicates of another record are removed before any
#     model is fitted, because a duplicate that falls on both sides of a split
#     is the most direct form of leakage there is.
#   * Skill is reported against the grand mean of the training partition, not
#     against the mean of the evaluation fold. The second choice makes the
#     coefficient of determination depend on how far a held-out study sits from
#     the overall mean, which is a property of the split rather than of the
#     model.
#   * Outer cross-validation is repeated with several fold assignments, and the
#     difference between protocols is tested fold by fold.
#
# LICENSE  MIT
# =============================================================================

import os, sys, io, time, math, json, warnings, subprocess
warnings.filterwarnings('ignore')

# ----------------------------- CONFIGURATION ---------------------------------
SEED        = 20250812     # one seed for everything
N_OUTER     = 5            # outer CV folds
N_INNER     = 3            # inner CV folds for hyper-parameter search
N_ITER      = 8            # randomised-search candidates per inner loop
N_REPEATS   = 3            # repeats of the outer cross-validation
DROP_EXACT_DUPLICATES = True   # remove records identical to another record
FAST_MODE   = False        # True = tiny settings for a smoke test (~2 min)
RUN_SHAP    = True
OUTDIR      = 'hydrosoil_v2_output'

if FAST_MODE:
    N_OUTER, N_INNER, N_ITER, N_REPEATS = 3, 2, 2, 1

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(f'{OUTDIR}/figures', exist_ok=True)

def _pip(pkg, mod=None):
    try:
        __import__(mod or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for p, m in [('xgboost', 'xgboost'), ('lightgbm', 'lightgbm'), ('shap', 'shap')]:
    _pip(p, m)

import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
from scipy import stats

from sklearn.experimental import enable_iterative_imputer          # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (BayesianRidge, Ridge, Lasso, ElasticNet,
                                  LinearRegression)
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, StackingRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.model_selection import GroupKFold, KFold, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone, BaseEstimator, RegressorMixin
from scipy.stats import spearmanr
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

plt.rcParams.update({'font.size': 9, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 150,
                     'savefig.bbox': 'tight', 'savefig.dpi': 300})
CLR = {'a': '#2f4b7c', 'b': '#f95d6a', 'c': '#665191', 'd': '#ffa600', 'g': '#8c8c8c'}


def banner(txt):
    print('\n' + '=' * 78 + f'\n{txt}\n' + '=' * 78, flush=True)


# ------------------------- CRASH-SAFE CHECKPOINTING --------------------------
# Long runs on Colab can be interrupted. Every expensive loop appends its result
# to a checkpoint file as soon as it finishes. If the session drops, just run the
# script again: completed work is skipped and the run continues where it stopped.
CKPT = f'{OUTDIR}/checkpoints'
os.makedirs(CKPT, exist_ok=True)

def ckpt_done(name, key):
    """Keys already completed for this checkpoint file."""
    f = f'{CKPT}/{name}.csv'
    return set(pd.read_csv(f)[key]) if os.path.exists(f) else set()

def ckpt_add(name, row):
    f = f'{CKPT}/{name}.csv'
    pd.DataFrame([row]).to_csv(f, mode='a', header=not os.path.exists(f), index=False)

def ckpt_load(name, key):
    return pd.read_csv(f'{CKPT}/{name}.csv').drop_duplicates(key)

def oof_save(store):
    joblib.dump(store, f'{CKPT}/oof.pkl')

def oof_load():
    return joblib.load(f'{CKPT}/oof.pkl') if os.path.exists(f'{CKPT}/oof.pkl') else {}


# =============================================================================
# PHASE 1 - LOAD THE RAW DATABASE
# =============================================================================
banner('PHASE 1 | DATA INGESTION')

# The database file is located in this order:
#   1. a path given on the command line          python script.py /path/SWIG.xlsx
#   2. the environment variable SWIG_PATH
#   3. any file whose name contains "swig" in the current folder, the script folder,
#      Downloads, Desktop, the home folder or /content  (Colab)
#   4. Colab upload widget, if running in Colab
#   5. a native "open file" dialog, if a desktop GUI is available
#   6. a typed path at the prompt
# Nothing about the file name or the sheet names is assumed beyond containing the
# words "swig", "meta" and "loc".

import glob as _glob

def _candidates():
    pats, seen = [], []
    here = os.getcwd()
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        script_dir = here
    home = os.path.expanduser('~')
    for folder in [here, script_dir, '/content', '/content/drive/MyDrive',
                   os.path.join(home, 'Downloads'), os.path.join(home, 'Desktop'),
                   os.path.join(home, 'Documents'), home]:
        if not os.path.isdir(folder):
            continue
        for f in _glob.glob(os.path.join(folder, '*.xlsx')) + \
                 _glob.glob(os.path.join(folder, '*.xls')):
            if 'swig' in os.path.basename(f).lower() and f not in seen:
                seen.append(f); pats.append(f)
    return pats

def locate_database():
    # 1 - command line
    for a in sys.argv[1:]:
        if a.lower().endswith(('.xlsx', '.xls')) and os.path.exists(a):
            return a
    # 2 - environment variable
    env = os.environ.get('SWIG_PATH')
    if env and os.path.exists(env):
        return env
    # 3 - search the usual folders
    hits = _candidates()
    if len(hits) == 1:
        print(f'Found database: {hits[0]}')
        return hits[0]
    if len(hits) > 1:
        print('Several candidate files found:')
        for i, h in enumerate(hits):
            print(f'  [{i}] {h}')
        try:
            k = int(input('Choose a number: ').strip())
            return hits[k]
        except Exception:
            return hits[0]
    # 4 - Colab upload
    try:
        from google.colab import files as _cf
        print('Select SWIG_database.xlsx from your computer:')
        up = _cf.upload()
        if up:
            return next(iter(up))
    except Exception:
        pass
    # 5 - desktop file dialog
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True)
        f = filedialog.askopenfilename(
            title='Select the SWIG database file',
            filetypes=[('Excel files', '*.xlsx *.xls'), ('All files', '*.*')])
        root.destroy()
        if f:
            return f
    except Exception:
        pass
    # 6 - typed path
    try:
        f = input('Full path to the SWIG database file: ').strip().strip('"\'')
        if os.path.exists(f):
            return f
    except Exception:
        pass
    sys.exit('No database file selected. Re-run and provide the SWIG .xlsx file.')

RAW_PATH = locate_database()
print(f'Using: {RAW_PATH}')

# --------- tolerant sheet and header detection --------------------------------
# Column names and header positions vary between SWIG releases and between
# manually edited copies, so nothing is hard-coded.

ALIASES = {
    'Code':          ['code', 'soil no', 'soil number', 'id', 'no'],
    'Clay':          ['clay', 'clay (%)', 'clay%'],
    'Silt':          ['silt', 'silt (%)', 'silt%'],
    'Sand':          ['sand', 'sand (%)', 'sand%'],
    'OC':            ['oc', 'organic carbon', 'soc', 'org c', 'oc (%)'],
    'Db':            ['db', 'bd', 'bulk density', 'rho b', 'bulk density (g/cm3)'],
    'Dp':            ['dp', 'particle density'],
    'Ksat':          ['ksat', 'ks', 'k sat', 'saturated hydraulic conductivity'],
    'dg':            ['dg', 'geometric mean diameter', 'd g'],
    'Sg':            ['sg', 'geometric standard deviation', 's g'],
    'Gravel':        ['gravel'],
    'WC_s':          ['wc_s', 'wcs', 'saturated water content', 'theta s'],
    'Instrument':    ['instrument', 'method', 'device', 'measurement method'],
    'Landuse_Code':  ['landuse_code', 'landuse code', 'land use code',
                      'landuse (classifed)', 'landuse (classified)', 'land use'],
    'Texture Class': ['texture class', 'textureclass', 'texture', 'usda texture'],
}

def _norm(c):
    return str(c).strip().lower().replace('_', ' ').replace('.', '')

def read_sheet_smart(xls, sheet, must_have):
    """Try several header rows and return the frame whose columns match best."""
    best, best_hits = None, -1
    for hdr in [1, 0, 2, 3]:
        try:
            d = pd.read_excel(xls, sheet_name=sheet, header=hdr)
        except Exception:
            continue
        d.columns = [str(c).strip() for c in d.columns]
        norm = {_norm(c) for c in d.columns}
        hits = sum(any(a in norm for a in ALIASES.get(k, [k.lower()]))
                   for k in must_have)
        if hits > best_hits:
            best, best_hits = d, hits
    if best is None:
        sys.exit(f'Could not read sheet "{sheet}".')
    return best

def standardise(d):
    """Rename recognised columns to the canonical names used below."""
    lookup = {_norm(c): c for c in d.columns}
    ren = {}
    for canon, alist in ALIASES.items():
        if canon in d.columns:
            continue
        for a in alist:
            if a in lookup:
                ren[lookup[a]] = canon
                break
    return d.rename(columns=ren)

xls = pd.ExcelFile(RAW_PATH)
print('Sheets found: ' + ', '.join(xls.sheet_names))

meta_sheet = next((sh for sh in xls.sheet_names if 'meta' in sh.lower()), None)
loc_sheet = next((sh for sh in xls.sheet_names if sh.lower().startswith('loc')), None)
if meta_sheet is None:
    meta_sheet = xls.sheet_names[0]
    print(f'No sheet named "Metadata"; using the first sheet: {meta_sheet}')

meta = standardise(read_sheet_smart(xls, meta_sheet,
                                    ['Clay', 'Sand', 'OC', 'Db', 'Ksat']))
print(f'Metadata sheet "{meta_sheet}": {len(meta)} rows, {meta.shape[1]} columns')

REQUIRED = ['Clay', 'Silt', 'Sand', 'OC', 'Db', 'Ksat', 'Code']
missing = [c for c in REQUIRED if c not in meta.columns]
if missing:
    sys.exit('The metadata sheet is missing required columns: ' + ', '.join(missing) +
             '\nColumns present: ' + ', '.join(map(str, meta.columns[:40])))

# optional columns: create them empty if the file does not carry them
for opt in ['Dp', 'dg', 'Sg', 'WC_s', 'Gravel', 'Instrument', 'Landuse_Code',
            'Texture Class']:
    if opt not in meta.columns:
        meta[opt] = np.nan
        print(f'  note: column "{opt}" absent, created empty')

NUMCOLS = ['Clay', 'Silt', 'Sand', 'OC', 'Db', 'Dp', 'Ksat', 'dg', 'Sg', 'WC_s',
           'Gravel', 'Instrument', 'Landuse_Code']
for c in NUMCOLS:
    meta[c] = pd.to_numeric(meta[c], errors='coerce')

# dg and Sg (Shirazi & Boersma 1984) are derived from texture if not supplied
if meta['dg'].notna().sum() == 0:
    print('  note: dg/Sg absent, deriving them from the texture fractions')
    D = {'Clay': 0.001, 'Silt': 0.026, 'Sand': 1.025}          # mm, class midpoints
    fr = meta[['Clay', 'Silt', 'Sand']].div(
        meta[['Clay', 'Silt', 'Sand']].sum(axis=1).replace(0, np.nan), axis=0)
    lg = sum(fr[k] * np.log(D[k]) for k in D)
    meta['dg'] = np.exp(lg)
    meta['Sg'] = np.exp(np.sqrt(
        sum(fr[k] * (np.log(D[k]) - lg) ** 2 for k in D).clip(lower=0)))

# ---- locations sheet: supplies the grouping variables ------------------------
if loc_sheet is None:
    sys.exit('No "Locations" sheet found. It supplies the study, country and '
             'coordinate labels that the grouped validation depends on.')

loc = read_sheet_smart(xls, loc_sheet, ['Code'])
loc = loc.iloc[:, :8]
loc.columns = (['DatasetID', 'From', 'To', 'Provider', 'Location', 'Country',
                'lat', 'lon'])[:loc.shape[1]]
print(f'Locations sheet "{loc_sheet}": {len(loc)} contributed blocks')

# =============================================================================
# PHASE 2 - SAMPLE DEFINITION WITH A FULL EXCLUSION FLOW
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 2 | SAMPLE DEFINITION
#
# A Ksat value is available for only a subset of the SWIG records, so the analysed
# sample is much smaller than the compilation. Every step from the full database
# to the final sample is recorded in a flow table, and no record is dropped
# silently.
#
# Exclusions applied, in order:
#   * no reported Ksat value, or Ksat <= 0 (the target is log-transformed)
#   * bulk density above the assumed particle density of 2.65 g/cm3, which is
#     physically impossible
#   * texture fractions that do not sum to 100 % within a 5 % tolerance
#   * no core soil property reported at all
#   * records whose texture, organic carbon, bulk density and Ksat are identical
#     to those of a record already retained
#
# The last exclusion matters for the question this study asks. Repeated runs
# reported by a contributing study appear in the compilation as identical rows.
# Under a random split the same row can land in both partitions, so part of any
# apparent skill is the model returning a value it has already seen. The
# duplicated sample is retained separately so that the size of that effect can
# be measured rather than assumed.
#
# The grouping variables are then reconstructed from the record-number ranges in
# the locations sheet. Each record receives its source dataset, data provider,
# country and coordinates. These labels are what make the grouped validation in
# PHASE 5 possible; without them the hierarchy of the compilation is invisible to
# the evaluation.
# -----------------------------------------------------------------------------

banner('PHASE 2 | SAMPLE DEFINITION (reported as a flow table)')

flow = [('Infiltration records in SWIG', len(meta))]
flow.append(('Records with a reported Ksat value', int(meta['Ksat'].notna().sum())))

df = meta[meta['Ksat'] > 0].copy()
flow.append(('Ksat > 0 (log-transformable)', len(df)))

n = int((df['Db'] > 2.65).sum())
df = df[~(df['Db'] > 2.65)]
flow.append((f'Excluded: bulk density > 2.65 g/cm3 (n={n})', len(df)))

tex = df[['Clay', 'Silt', 'Sand']]
bad = (~tex.isna().any(axis=1)) & (tex.sum(axis=1).sub(100).abs() > 5)
flow.append((f'Excluded: texture not summing to 100 +/- 5 % (n={int(bad.sum())})',
             len(df := df[~bad])))

core = ['Clay', 'Silt', 'Sand', 'OC', 'Db']
n = int(df[core].isna().all(axis=1).sum())
df = df[~df[core].isna().all(axis=1)]
flow.append((f'Excluded: no core soil property reported (n={n})', len(df)))

DUP_KEY = ['Clay', 'Silt', 'Sand', 'OC', 'Db', 'Ksat']
n_dup = int(df.duplicated(subset=DUP_KEY, keep='first').sum())
flow.append((f'Excluded: exact duplicate of a retained record (n={n_dup})',
             len(df) - (n_dup if DROP_EXACT_DUPLICATES else 0)))
flow.append(('FINAL ANALYSIS SET', len(df) - (n_dup if DROP_EXACT_DUPLICATES else 0)))

tab_flow = pd.DataFrame(flow, columns=['Step', 'Records remaining'])
print(tab_flow.to_string(index=False))

# ---- grouping variables from the Locations sheet ----------------------------
loc['From'] = pd.to_numeric(loc['From'], errors='coerce')
loc['To'] = pd.to_numeric(loc['To'], errors='coerce')
loc = loc.dropna(subset=['From', 'To'])
loc[['From', 'To']] = loc[['From', 'To']].astype(int)

def _lookup(code):
    hit = loc[(loc['From'] <= code) & (loc['To'] >= code)]
    if len(hit) == 0:
        return pd.Series({'DatasetID': np.nan, 'Provider': np.nan,
                          'Country': np.nan, 'lat': np.nan, 'lon': np.nan})
    r = hit.iloc[0]
    return pd.Series({'DatasetID': r['DatasetID'], 'Provider': r['Provider'],
                      'Country': r['Country'], 'lat': r['lat'], 'lon': r['lon']})

df = pd.concat([df.reset_index(drop=True),
                df['Code'].apply(_lookup).reset_index(drop=True)], axis=1)
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
df['GeoBlock'] = (np.floor(df['lat'] / 5).astype('Int64').astype(str) + '_' +
                  np.floor(df['lon'] / 5).astype('Int64').astype(str))
df['Instrument'] = df['Instrument'].fillna(-1).astype(int)
df['Landuse_Code'] = df['Landuse_Code'].fillna(-1).astype(int)
df['logKsat'] = np.log10(df['Ksat'])

# the sample with the duplicated records still in it is kept so that their
# contribution can be measured in Phase 5b; the analysis itself uses the
# deduplicated sample
df_with_duplicates = df.copy()
if DROP_EXACT_DUPLICATES:
    df = df[~df.duplicated(subset=DUP_KEY, keep='first')].reset_index(drop=True)
print(f'analysis set: {len(df)} records after removing {n_dup} exact duplicates')

tab_groups = pd.DataFrame({
    'Grouping variable': ['Source dataset', 'Data provider', 'Country',
                          '5 deg geographic block', 'Measurement instrument'],
    'Levels': [df['DatasetID'].nunique(), df['Provider'].nunique(),
               df['Country'].nunique(), df['GeoBlock'].nunique(),
               df['Instrument'].nunique()],
    'Largest level (n)': [df['DatasetID'].value_counts().max(),
                          df['Provider'].value_counts().max(),
                          df['Country'].value_counts().max(),
                          df['GeoBlock'].value_counts().max(),
                          df['Instrument'].value_counts().max()]})
print('\n' + tab_groups.to_string(index=False))

tab_missing = pd.DataFrame({'Variable': core + ['dg', 'Sg', 'Dp']})
tab_missing['Missing (%)'] = [round(100 * df[v].isna().mean(), 1)
                              for v in tab_missing['Variable']]
print('\nPredictor missingness:\n' + tab_missing.to_string(index=False))


# =============================================================================
# PHASE 3 - PHYSICS-GUIDED FEATURES (every equation, constant and unit stated)
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 3 | PHYSICS-GUIDED FEATURE CONSTRUCTION
#
# Deterministic transformations of the measured inputs, motivated by soil physics.
# Each is defined below with its equation, its constants and its units, so that
# the construction can be reproduced or challenged.
#
# These are physics-GUIDED features: they add domain knowledge to the inputs. They
# do not impose conservation laws, governing equations or admissibility
# constraints on the model output, and the script therefore avoids the term
# physics-informed, which is reserved for constrained learning schemes.
#
# Constructions:
#   Porosity          phi = 1 - rho_b / rho_p, with rho_p fixed at 2.65 g/cm3
#   Porosity_sq       phi squared, for a non-linear response to pore volume
#   PoreRatio         phi^3 / (1 - phi)^2, the Kozeny-Carman porosity group
#   BD_Adams          Adams (1973) two-phase mixing model for bulk density
#   RelCompaction     measured / theoretical bulk density: compaction that is not
#                     explained by organic-matter content
#   OrganicPore       organic carbon multiplied by porosity
#   log_dg, Sg_       geometric mean particle diameter and its geometric standard
#                     deviation (Shirazi and Boersma 1984); derived from texture
#                     if the database does not supply them
#   KC_logK           Kozeny-Carman conductivity, dimensionally explicit and
#                     converted to cm/h. Used both as a predictor and, in PHASE 7,
#                     as a completely unfitted baseline
#   Sand_Clay         sand-to-clay ratio, a coarseness index
#
# Six feature sets are assembled so that the contribution of each group of inputs
# can be isolated later: texture only, standard PTF inputs, standard plus
# physics-guided, that set plus measurement metadata, the four inputs the field
# tool asks for, and metadata alone.
# -----------------------------------------------------------------------------

banner('PHASE 3 | PHYSICS-GUIDED FEATURE CONSTRUCTION')

RHO_P   = 2.65      # g/cm3   assumed particle density (quartz-dominated)
RHO_OM  = 0.224     # g/cm3   bulk density of organic matter (Adams 1973)
RHO_MIN = 1.46      # g/cm3   mineral bulk density (Adams 1973)
VB      = 1.724     # -       van Bemmelen factor, OM = 1.724 x OC
G       = 9.81      # m/s2
RHO_W   = 1000.0    # kg/m3
MU_W    = 1.002e-3  # Pa s    dynamic viscosity of water at 20 C

def add_physics(d):
    d = d.copy()
    f = (VB * d['OC']) / 100.0                                   # OM mass fraction
    # Adams (1973) two-phase mixing model for bulk density, g/cm3
    d['BD_Adams']      = 1.0 / (f / RHO_OM + (1 - f) / RHO_MIN)
    # measured / theoretical bulk density: compaction independent of organic content
    d['RelCompaction'] = d['Db'] / d['BD_Adams']
    d['Porosity']      = 1.0 - d['Db'] / RHO_P
    d['Porosity_sq']   = d['Porosity'] ** 2
    d['PoreRatio']     = d['Porosity'] ** 3 / np.maximum((1 - d['Porosity']) ** 2, 1e-6)
    d['OrganicPore']   = d['OC'] * d['Porosity']
    # geometric mean particle diameter, mm (Shirazi & Boersma 1984; supplied by SWIG)
    d['log_dg']        = np.log10(np.maximum(d['dg'], 1e-6))
    d['Sg_']           = d['Sg']
    # Kozeny-Carman, dimensionally explicit, converted to cm/h
    de_m  = d['dg'] / 1000.0
    K_ms  = (RHO_W * G / MU_W) * (de_m ** 2 / 180.0) * d['PoreRatio']
    d['KC_logK']       = np.log10(np.maximum(K_ms * 100 * 3600, 1e-12))
    d['Sand_Clay']     = d['Sand'] / np.maximum(d['Clay'], 1.0)
    return d

df = add_physics(df)

BASE_TEX = ['Clay', 'Silt', 'Sand']
BASE_PTF = ['Clay', 'Silt', 'Sand', 'OC', 'Db']
PHYS     = ['Porosity', 'Porosity_sq', 'PoreRatio', 'BD_Adams', 'RelCompaction',
            'OrganicPore', 'log_dg', 'Sg_', 'KC_logK', 'Sand_Clay']
METAD    = ['Instrument', 'Landuse_Code']
DEPLOY   = ['Clay', 'Sand', 'OC', 'Db']

FEATURE_SETS = {
    'F1 Texture only':              BASE_TEX,
    'F2 Standard PTF inputs':       BASE_PTF,
    'F3 PTF + physics-guided':      BASE_PTF + PHYS,
    'F4 F3 + measurement metadata': BASE_PTF + PHYS + METAD,
    'F5 Four-input field set':      DEPLOY,
    'F6 Metadata only':             METAD,
}
print('feature sets: ' + ', '.join(f'{k} ({len(v)})' for k, v in FEATURE_SETS.items()))

y         = df['logKsat'].values
groups_ds = df['DatasetID'].fillna(-1).astype(int).values
groups_ct = df['Country'].fillna('NA').values
groups_gb = df['GeoBlock'].fillna('NA').values
groups_in = df['Instrument'].values
SD_Y      = float(df['logKsat'].std())

def get_X(cols):
    return df[cols].astype(float).values


# =============================================================================
# PHASE 4 - EVALUATION MACHINERY
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 4 | EVALUATION MACHINERY
#
# Everything the experiments share: fold-safe imputation, the model panel, the
# metrics and the split generators.
#
# nested_cv() is the core routine. For each outer fold it imputes using the
# training rows only, searches the hyper-parameter space on a group-aware inner
# split of those same training rows, refits the best configuration, and predicts
# the held-out rows. Model selection therefore never touches the data used to
# report performance.
#
# Metrics are reported per fold and summarised with a 95 % confidence interval
# across folds:
#   R2, RMSE, MAE   on the log10 scale
#   Bias            mean signed error, which detects systematic level shifts
#   MedFactor       typical multiplicative error on the original scale, that is
#                   the factor by which a prediction misses
#   Within10        percentage of predictions within one order of magnitude
#
# The original-scale metrics matter because a modest RMSE in log units still
# corresponds to a large error in cm/h, which is what an engineer acts on.
#
# The Gaussian process is fitted on a bounded subsample because its cost grows
# with the cube of the training size; every other model uses all training rows.
# -----------------------------------------------------------------------------

banner('PHASE 4 | NESTED, GROUPED, PREPROCESSING-SAFE CROSS-VALIDATION')

def impute_fold(Xtr, Xte):
    """Multivariate imputation by chained equations, fitted on the training rows
    only and then applied to the held-out rows. Fitting an imputer on the full
    table before splitting lets information from the evaluation set influence the
    training set, which inflates every downstream score."""
    if not np.isnan(Xtr).any() and not np.isnan(Xte).any():
        return Xtr, Xte
    im = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=SEED)
    return im.fit_transform(Xtr), im.transform(Xte)

def mk(model, scale=True):
    return Pipeline(([('sc', StandardScaler())] if scale else []) + [('m', model)])

class SubsampleGP(BaseEstimator, RegressorMixin):
    """Gaussian process on at most `cap` training rows, for tractability."""
    def __init__(self, cap=700, seed=SEED):
        self.cap, self.seed = cap, seed
    def fit(self, X, y):
        rs = np.random.RandomState(self.seed)
        i = rs.choice(len(X), min(self.cap, len(X)), replace=False)
        self.gp_ = GaussianProcessRegressor(
            kernel=ConstantKernel() * RBF() + WhiteKernel(), normalize_y=True,
            random_state=self.seed, alpha=1e-6, n_restarts_optimizer=0).fit(X[i], y[i])
        return self
    def predict(self, X):
        return self.gp_.predict(X)

NTREE = 100 if FAST_MODE else 400
MODELS = {
 'Extra trees': (mk(ExtraTreesRegressor(random_state=SEED, n_jobs=-1), False),
   {'m__n_estimators': [NTREE], 'm__max_features': [0.3, 0.5, 0.8, 1.0],
    'm__min_samples_leaf': [1, 2, 4, 8], 'm__max_depth': [None, 12, 20]}),
 'Random forest': (mk(RandomForestRegressor(random_state=SEED, n_jobs=-1), False),
   {'m__n_estimators': [NTREE], 'm__max_features': [0.3, 0.5, 0.8],
    'm__min_samples_leaf': [1, 2, 4, 8], 'm__max_depth': [None, 12, 20]}),
 'XGBoost': (mk(XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0), False),
   {'m__n_estimators': [300, 600], 'm__learning_rate': [0.02, 0.05, 0.1],
    'm__max_depth': [3, 4, 6], 'm__subsample': [0.7, 1.0],
    'm__colsample_bytree': [0.6, 0.8, 1.0], 'm__reg_lambda': [1, 5, 20]}),
 'LightGBM': (mk(LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1), False),
   {'m__n_estimators': [300, 600], 'm__learning_rate': [0.02, 0.05, 0.1],
    'm__num_leaves': [15, 31, 63], 'm__min_child_samples': [5, 20, 40],
    'm__subsample': [0.7, 1.0], 'm__colsample_bytree': [0.6, 0.8, 1.0]}),
 'Gradient boosting': (mk(GradientBoostingRegressor(random_state=SEED), False),
   {'m__n_estimators': [200, 400], 'm__learning_rate': [0.02, 0.05, 0.1],
    'm__max_depth': [2, 3, 4], 'm__subsample': [0.7, 1.0]}),
 'Regression tree': (mk(DecisionTreeRegressor(random_state=SEED), False),
   {'m__max_depth': [4, 6, 10, None], 'm__min_samples_leaf': [2, 5, 10, 20]}),
 'k-NN (distance)': (mk(KNeighborsRegressor(weights='distance')),
   {'m__n_neighbors': [3, 5, 10, 20], 'm__p': [1, 2]}),
 'SVR (RBF)': (mk(SVR()),
   {'m__C': [1, 10, 100], 'm__gamma': ['scale', 0.05, 0.1], 'm__epsilon': [0.1, 0.3]}),
 'Neural network (MLP)': (mk(MLPRegressor(random_state=SEED, max_iter=800,
                                          early_stopping=True)),
   {'m__hidden_layer_sizes': [(64,), (128, 64), (64, 32)],
    'm__alpha': [1e-4, 1e-2, 1e-1], 'm__learning_rate_init': [1e-3, 5e-3]}),
 'Gaussian process': (mk(SubsampleGP()), {}),
 'Ridge': (mk(Ridge(random_state=SEED)), {'m__alpha': [0.1, 1, 10, 100]}),
 'Lasso': (mk(Lasso(random_state=SEED, max_iter=5000)),
           {'m__alpha': [0.001, 0.01, 0.1, 1]}),
 'Elastic net': (mk(ElasticNet(random_state=SEED, max_iter=5000)),
           {'m__alpha': [0.001, 0.01, 0.1, 1], 'm__l1_ratio': [0.2, 0.5, 0.8]}),
 'Multiple linear regression': (mk(LinearRegression()), {}),
 'Stacking ensemble': (mk(StackingRegressor(
     estimators=[('et', ExtraTreesRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)),
                 ('lgb', LGBMRegressor(n_estimators=300, random_state=SEED, verbose=-1)),
                 ('knn', KNeighborsRegressor(n_neighbors=10, weights='distance'))],
     final_estimator=Ridge(), cv=3, n_jobs=1)), {}),
}

def metrics(yt, yp, y_train=None):
    """Performance of a set of predictions.

    R2 is the conventional coefficient of determination, computed against the
    mean of the evaluation fold. Skill is computed against the mean of the
    training partition, which is the prediction a model with no information
    would actually make. Skill is the quantity to compare across protocols,
    because it does not move when a held-out study happens to sit far from the
    overall mean, whereas R2 does. RMSE, MAE and Bias are in log10 units.
    MedFactor is the typical multiplicative error on the original scale, and
    Within10 the percentage of predictions within one order of magnitude.
    """
    e = yp - yt
    out = dict(R2=r2_score(yt, yp),
               RMSE=float(np.sqrt(mean_squared_error(yt, yp))),
               MAE=float(mean_absolute_error(yt, yp)),
               Bias=float(np.mean(e)),
               MedFactor=float(10 ** np.median(np.abs(e))),
               Within10=float(100 * np.mean(np.abs(e) <= 1.0)))
    if y_train is not None and len(y_train):
        sse_model = float(np.sum(e ** 2))
        sse_null = float(np.sum((yt - float(np.mean(y_train))) ** 2))
        out['Skill'] = 1.0 - sse_model / sse_null if sse_null > 0 else np.nan
        out['RMSE_null'] = float(np.sqrt(sse_null / len(yt)))
    return out

def ci95(v):
    v = np.asarray(v, float)
    return 1.96 * v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan

def splits_group(g, n=N_OUTER, repeats=1, seed=SEED):
    """Grouped folds. With repeats greater than one the groups are reassigned to
    folds under a different random permutation each time, so that the spread of
    the result over fold assignments can be reported."""
    out = []
    for r in range(repeats):
        if r == 0:
            out += list(GroupKFold(n_splits=n).split(np.zeros(len(g)), groups=g))
        else:
            rs = np.random.RandomState(seed + r)
            ug = pd.unique(g); rs.shuffle(ug)
            bucket = {gr: i % n for i, gr in enumerate(ug)}
            lab = np.array([bucket[x] for x in g])
            out += [(np.where(lab != k)[0], np.where(lab == k)[0]) for k in range(n)]
    return out

def splits_random(n=N_OUTER, seed=SEED, repeats=1):
    """Row-wise folds, optionally repeated with a different shuffle."""
    out = []
    for r in range(repeats):
        out += list(KFold(n_splits=n, shuffle=True,
                          random_state=seed + r).split(np.zeros(len(y))))
    return out

def nested_cv(model, grid, X, yv, groups, splits, n_iter=N_ITER):
    rows, oof = [], np.full(len(yv), np.nan)
    for k, (tr, te) in enumerate(splits):
        Xtr, Xte = impute_fold(X[tr], X[te])
        if grid:
            inner = list(GroupKFold(n_splits=N_INNER).split(Xtr, yv[tr], groups[tr]))
            s = RandomizedSearchCV(model, grid, n_iter=n_iter, cv=inner,
                                   scoring='neg_root_mean_squared_error',
                                   random_state=SEED, n_jobs=-1, refit=True)
            s.fit(Xtr, yv[tr]); best = s.best_estimator_
        else:
            best = clone(model).fit(Xtr, yv[tr])
        yp = best.predict(Xte); oof[te] = yp
        m = metrics(yv[te], yp, y_train=yv[tr]); m['fold'] = k
        rows.append(m)
    return pd.DataFrame(rows), oof


# =============================================================================
# PHASE 5 - EXPERIMENT 1: DOES THE VALIDATION PROTOCOL CHANGE THE ANSWER?
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 5 | EXPERIMENT 1 - VALIDATION PROTOCOL COMPARISON
#
# The central experiment. One model, three feature sets, four ways of assigning
# records to folds:
#
#   P1  random row-wise      records assigned at random, ignoring the hierarchy
#   P2  grouped by dataset   every evaluated record belongs to a source study that
#                            was absent from training
#   P3  grouped by country
#   P4  grouped by 5-degree geographic block
#
# Everything except the fold-assignment rule is held constant, so any difference
# in the reported skill is attributable to the evaluation design alone. The outer
# cross-validation is repeated with several fold assignments, and the difference
# between the row-wise and the grouped protocol is tested fold by fold with a
# paired t test and a bootstrap interval, so that the size of the difference is
# reported with an uncertainty rather than as a single number.
#
# The phase then decomposes the outcome. A negative R2 can mean two different
# things: the model may rank soils correctly but place the whole study at the
# wrong level, or it may lose both. Removing the study mean from the predictions
# and the observations separates the two, and distinguishes a calibration problem
# that a local offset could repair from a complete failure to transfer.
# -----------------------------------------------------------------------------

banner('PHASE 5 | EXPERIMENT 1 - VALIDATION PROTOCOL COMPARISON')

t0 = time.time()
PROTOCOLS = [
    ('P1 Random row-wise', splits_random(repeats=N_REPEATS), groups_ds),
    ('P2 Grouped by dataset', splits_group(groups_ds, repeats=N_REPEATS), groups_ds),
    ('P3 Grouped by country', splits_group(groups_ct, repeats=N_REPEATS), groups_ct),
    ('P4 Grouped by geo block', splits_group(groups_gb, repeats=N_REPEATS), groups_gb),
]
FS_MAIN = ['F2 Standard PTF inputs', 'F3 PTF + physics-guided',
           'F4 F3 + measurement metadata']

rows, oof_store, folds_store = [], {}, {}
mdl, grid = MODELS['Extra trees']
for pname, sp, grp in PROTOCOLS:
    for fs in FS_MAIN:
        r, oof = nested_cv(mdl, grid, get_X(FEATURE_SETS[fs]), y, grp, sp)
        oof_store[f'{pname}|{fs}'] = oof
        folds_store[f'{pname}|{fs}'] = r
        rows.append(dict(Protocol=pname, Features=fs,
                         R2=r.R2.mean(), R2ci=ci95(r.R2),
                         Skill=r.Skill.mean(), Skillci=ci95(r.Skill),
                         RMSE=r.RMSE.mean(), RMSEci=ci95(r.RMSE),
                         RMSE_null=r.RMSE_null.mean(),
                         MAE=r.MAE.mean(), Bias=r.Bias.mean(),
                         MedFactor=r.MedFactor.mean(), Within10=r.Within10.mean(),
                         n_folds=len(r)))
        print(f'{pname:26s} {fs:30s} R2={r.R2.mean():+.3f}  Skill={r.Skill.mean():+.3f} '
              f'RMSE={r.RMSE.mean():.3f} (null {r.RMSE_null.mean():.3f}) '
              f'[{time.time()-t0:.0f}s]', flush=True)
tab_protocols = pd.DataFrame(rows).round(3)
print('\n' + tab_protocols.to_string(index=False))
print(f'\nStandard deviation of the target: {SD_Y:.3f} log10 units. Skill is measured '
      f'against the mean of the training partition, so a value of zero means the model '
      f'is no better than that mean and a negative value means it is worse.')

# ---- is the difference between protocols larger than the fold-to-fold spread?
comp = []
for fs in FS_MAIN:
    a = folds_store[f'P1 Random row-wise|{fs}']
    for pname in ['P2 Grouped by dataset', 'P3 Grouped by country',
                  'P4 Grouped by geo block']:
        b = folds_store[f'{pname}|{fs}']
        n = min(len(a), len(b))
        for metric in ['R2', 'Skill', 'RMSE']:
            d = a[metric].values[:n] - b[metric].values[:n]
            t_stat, p_val = stats.ttest_rel(a[metric].values[:n], b[metric].values[:n])
            rs = np.random.RandomState(SEED)
            boot = [np.mean(rs.choice(d, len(d), replace=True)) for _ in range(5000)]
            comp.append(dict(Features=fs, Protocol=pname, Metric=metric,
                             Difference=float(np.mean(d)),
                             CI_low=float(np.percentile(boot, 2.5)),
                             CI_high=float(np.percentile(boot, 97.5)),
                             t=float(t_stat), p=float(p_val), n_folds=n))
tab_protocol_test = pd.DataFrame(comp).round(4)
print('\nDifference between the row-wise protocol and each grouped protocol,')
print('paired over folds, with a bootstrap interval:')
print(tab_protocol_test[tab_protocol_test.Metric.isin(['Skill', 'R2'])].to_string(index=False))

# ---- decomposition: level bias vs within-study ranking ----------------------
rows = []
for key, yp in oof_store.items():
    proto, fs = key.split('|')
    ok = ~np.isnan(yp)
    d = df.loc[ok].copy(); d['yt'] = y[ok]; d['yh'] = yp[ok]
    a = d.groupby('DatasetID')[['yt', 'yh']].transform(lambda s: s - s.mean())
    big = d.groupby('DatasetID')['yt'].transform('size') >= 20
    rows.append(dict(Protocol=proto, Features=fs,
                     R2_pooled=r2_score(d['yt'], d['yh']),
                     Spearman_pooled=spearmanr(d['yt'], d['yh']).statistic,
                     R2_within_study=r2_score(a['yt'][big], a['yh'][big]),
                     Spearman_within_study=spearmanr(a['yt'][big], a['yh'][big]).statistic,
                     MeanAbsStudyBias=float(d.groupby('DatasetID')
                                            .apply(lambda g: (g['yh'] - g['yt']).mean())
                                            .abs().mean())))
tab_decomp = pd.DataFrame(rows).round(3)
print('\nSkill / bias decomposition:\n' + tab_decomp.to_string(index=False))


# =============================================================================
# PHASE 5b - HOW MUCH OF THE ROW-WISE RESULT COMES FROM DUPLICATED RECORDS?
# =============================================================================
# -----------------------------------------------------------------------------
# The analysis set excludes records that are identical to another record. The
# same comparison is run on the sample with those records restored, so that the
# contribution of the most literal form of leakage can be separated from the
# study-level similarity that grouping addresses. Nothing else changes.
# -----------------------------------------------------------------------------
banner('PHASE 5b | CONTRIBUTION OF DUPLICATED RECORDS')

dup_rows = []
if DROP_EXACT_DUPLICATES and len(df_with_duplicates) > len(df):
    dfd = add_physics(df_with_duplicates)
    yd = dfd['logKsat'].values
    gd = dfd['DatasetID'].fillna(-1).astype(int).values
    Xd = dfd[FEATURE_SETS['F3 PTF + physics-guided']].astype(float).values
    for pname, sp in [('P1 Random row-wise',
                       list(KFold(n_splits=N_OUTER, shuffle=True,
                                  random_state=SEED).split(np.zeros(len(yd))))),
                      ('P2 Grouped by dataset',
                       list(GroupKFold(n_splits=N_OUTER).split(np.zeros(len(yd)),
                                                               groups=gd)))]:
        r, _ = nested_cv(mdl, grid, Xd, yd, gd, sp)
        dup_rows.append(dict(Sample='Duplicates retained', n=len(yd), Protocol=pname,
                             R2=r.R2.mean(), Skill=r.Skill.mean(), RMSE=r.RMSE.mean()))
        print(f'duplicates retained  {pname:24s} R2={r.R2.mean():+.3f} '
              f'Skill={r.Skill.mean():+.3f}', flush=True)
    for pname in ['P1 Random row-wise', 'P2 Grouped by dataset']:
        r = folds_store[f'{pname}|F3 PTF + physics-guided']
        dup_rows.append(dict(Sample='Duplicates removed', n=len(y), Protocol=pname,
                             R2=r.R2.mean(), Skill=r.Skill.mean(), RMSE=r.RMSE.mean()))
tab_duplicates = pd.DataFrame(dup_rows).round(3)
if len(tab_duplicates):
    print('\n' + tab_duplicates.to_string(index=False))

# =============================================================================
# PHASE 6 - EXPERIMENT 2: FIFTEEN ALGORITHMS UNDER BOTH PROTOCOLS
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 6 | EXPERIMENT 2 - ALGORITHM BENCHMARK
#
# Fifteen algorithms spanning tree ensembles, boosting, instance-based learning,
# kernel methods, neural networks, Gaussian processes, penalised linear models and
# a stacking ensemble. Each is evaluated twice with identical nested searches:
# once with the row-wise protocol and once with the dataset-grouped protocol.
#
# The difference between the two, reported as Optimism, shows whether the loss of
# skill under grouped validation is a property of one model family or of the data
# structure. Running the same panel under both protocols also reveals whether the
# ranking of algorithms is stable, which determines whether a benchmark carried
# out on a row-wise split can be used to choose a model at all.
#
# No model is designated a winner. Under an evaluation this uncertain, a ranking
# by a single point estimate would not be meaningful.
# -----------------------------------------------------------------------------

banner('PHASE 6 | EXPERIMENT 2 - ALGORITHM BENCHMARK')

Xf3, sp_g, sp_r = get_X(FEATURE_SETS['F3 PTF + physics-guided']), \
                  splits_group(groups_ds), splits_random()
done = ckpt_done('algorithms', 'Algorithm')
for name, (m, g) in MODELS.items():
    if name in done:
        print(f'{name:28s} [checkpoint]', flush=True)
        continue
    try:
        rg, _ = nested_cv(m, g, Xf3, y, groups_ds, sp_g)
        rr, _ = nested_cv(m, g, Xf3, y, groups_ds, sp_r)
        ckpt_add('algorithms', dict(Algorithm=name, R2_grouped=rg.R2.mean(),
                 R2_grouped_ci=ci95(rg.R2), RMSE_grouped=rg.RMSE.mean(),
                 MedFactor_grouped=rg.MedFactor.mean(), R2_random=rr.R2.mean(),
                 Optimism=rr.R2.mean() - rg.R2.mean()))
        print(f'{name:28s} grouped {rg.R2.mean():+.3f} | row-wise {rr.R2.mean():+.3f} '
              f'| optimism {rr.R2.mean()-rg.R2.mean():+.3f} '
              f'[{time.time()-t0:.0f}s]', flush=True)
    except Exception as e:
        print(f'{name:28s} FAILED: {e!r}', flush=True)
tab_alg = ckpt_load('algorithms', 'Algorithm').sort_values(
    'R2_grouped', ascending=False).round(3)


# =============================================================================
# PHASE 7 - EXPERIMENT 3: FEATURE ABLATION AGAINST NON-LEARNED BASELINES
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 7 | EXPERIMENT 3 - FEATURE-SET ABLATION
#
# Whether an engineered feature helps is an empirical question, and answering it
# requires the same model evaluated under the same protocol with and without the
# feature. All six feature sets are therefore run through the identical grouped
# nested cross-validation.
#
# Two references that involve no learning are added so that the learned models can
# be judged against something:
#
#   B0  grand mean of the training partition, the null model. Any model that does
#       not beat it has no usable skill.
#   B1  Kozeny-Carman evaluated analytically with no fitting whatsoever. This
#       tests whether classical capillary-bundle theory, applied directly,
#       describes field-scale infiltration.
#
# Both baselines are computed fold-wise on the same splits as the models, so every
# row of the resulting table is comparable.
# -----------------------------------------------------------------------------

banner('PHASE 7 | EXPERIMENT 3 - FEATURE-SET ABLATION')

done = ckpt_done('ablation', 'FeatureSet')
for fs, cols in FEATURE_SETS.items():
    if fs in done:
        print(f'{fs:32s} [checkpoint]', flush=True)
        continue
    r, _ = nested_cv(mdl, grid, get_X(cols), y, groups_ds, sp_g)
    ckpt_add('ablation', dict(FeatureSet=fs, Inputs=len(cols), R2=r.R2.mean(),
             R2ci=ci95(r.R2), RMSE=r.RMSE.mean(), MAE=r.MAE.mean(),
             Bias=r.Bias.mean(), MedFactor=r.MedFactor.mean(),
             Within10=r.Within10.mean()))
    print(f'{fs:32s} R2={r.R2.mean():+.3f} RMSE={r.RMSE.mean():.3f} '
          f'[{time.time()-t0:.0f}s]', flush=True)
rows = ckpt_load('ablation', 'FeatureSet').to_dict('records')

msk = df['KC_logK'].notna().values
kc = metrics(y[msk], df['KC_logK'].values[msk])
kc.update(FeatureSet='B1 Kozeny-Carman (unfitted)', Inputs=2)
mp = np.full(len(y), np.nan)
for tr, te in sp_g:
    mp[te] = y[tr].mean()
b0 = metrics(y, mp); b0.update(FeatureSet='B0 Grand mean (null model)', Inputs=0)
tab_abl = pd.concat([pd.DataFrame(rows), pd.DataFrame([kc, b0])], ignore_index=True)
tab_abl = tab_abl[['FeatureSet', 'Inputs', 'R2', 'R2ci', 'RMSE', 'MAE', 'Bias',
                   'MedFactor', 'Within10']].round(3)
print('\n' + tab_abl.to_string(index=False))


# =============================================================================
# PHASE 8 - EXPERIMENT 4: TRANSFER TESTS AND VARIANCE COMPONENTS
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 8 | EXPERIMENT 4 - TRANSFER TESTS AND VARIANCE COMPONENTS
#
# Two targeted hold-out designs that probe specific confounders.
#
# Leave-one-instrument-out withholds every record obtained with a given device.
# Infiltration is measured with tension infiltrometers, single and double rings,
# pressure infiltrometers and rainfall simulators, which sample different pore-size
# ranges and impose different boundary conditions. If performance collapses when
# an instrument is withheld, part of what the model learned is measurement physics
# rather than soil physics.
#
# Leave-one-country-out withholds an entire region and asks the practical
# question a user faces: how does the model behave for a place it has never seen?
#
# The variance components then quantify, descriptively, how much of the variation
# in log10 Ksat is associated with each stratification. Comparing the share
# carried by the source dataset and the instrument with the share carried by
# texture class shows how much of the signal in the compilation is structural
# rather than pedological.
# -----------------------------------------------------------------------------

banner('PHASE 8 | EXPERIMENT 4 - LEAVE-ONE-INSTRUMENT / COUNTRY OUT')

ET_FIXED = mk(ExtraTreesRegressor(n_estimators=NTREE, max_features=0.5,
                                  min_samples_leaf=2, random_state=SEED, n_jobs=-1), False)

def holdout(X, gvar, min_n):
    out = []
    for lv in pd.Series(gvar).value_counts().index:
        te = np.where(gvar == lv)[0]
        if len(te) < min_n:
            continue
        tr = np.where(gvar != lv)[0]
        Xtr, Xte = impute_fold(X[tr], X[te])
        m = clone(ET_FIXED).fit(Xtr, y[tr])
        r = metrics(y[te], m.predict(Xte)); r.update(Level=str(lv), n=len(te))
        out.append(r)
    return pd.DataFrame(out)

X3 = get_X(FEATURE_SETS['F3 PTF + physics-guided'])
tab_inst = holdout(X3, groups_in.astype(int), 25).round(3)
tab_ctry = holdout(X3, groups_ct, 40).round(3)
print('Leave-one-instrument-out:\n' + tab_inst.to_string(index=False))
print('\nLeave-one-country-out:\n' + tab_ctry.to_string(index=False))

def var_share(col):
    gm = df.groupby(col)['logKsat']
    grand = df['logKsat'].mean()
    ssb = float((gm.size() * (gm.mean() - grand) ** 2).sum())
    sst = float(((df['logKsat'] - grand) ** 2).sum())
    return round(100 * ssb / sst, 1)

tab_var = pd.DataFrame({
    'Stratification': ['Source dataset', 'Country', 'Measurement instrument',
                       'Texture class', 'Land-use class'],
    'Share of variance in log10 Ksat (%)':
        [var_share('DatasetID'), var_share('Country'), var_share('Instrument'),
         var_share('Texture Class'), var_share('Landuse_Code')]})
print('\n' + tab_var.to_string(index=False))


# =============================================================================
# PHASE 9 - EXPERIMENT 5: OUT-OF-FOLD IMPORTANCE (and training-set SHAP)
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 9 | EXPERIMENT 5 - INTERPRETATION
#
# Permutation importance is computed on held-out folds: each feature is shuffled
# in the evaluation partition and the increase in RMSE is recorded. This measures
# how much a feature contributes to prediction for unseen studies.
#
# A negative value is informative rather than anomalous. It means that destroying
# the relationship the model learned for that feature improves prediction on new
# data, that is, the relationship was specific to the training studies.
#
# SHAP values are also computed, but on the fitted model and its training data.
# They describe what the model uses internally, which is not the same question. The
# two are reported side by side precisely because they can disagree: a training-set
# attribution that looks physically sensible is not evidence that the model has
# learned transferable physics.
# -----------------------------------------------------------------------------

banner('PHASE 9 | EXPERIMENT 5 - OUT-OF-FOLD PERMUTATION IMPORTANCE')

cols4 = FEATURE_SETS['F4 F3 + measurement metadata']; X4 = get_X(cols4)
imp = {c: [] for c in cols4}
for k, (tr, te) in enumerate(sp_g):
    Xtr, Xte = impute_fold(X4[tr], X4[te])
    m = clone(ET_FIXED).fit(Xtr, y[tr])
    base = np.sqrt(np.mean((m.predict(Xte) - y[te]) ** 2))
    rs = np.random.RandomState(SEED + k)
    for j, c in enumerate(cols4):
        d = []
        for _ in range(3 if FAST_MODE else 8):
            Xp = Xte.copy(); Xp[:, j] = rs.permutation(Xp[:, j])
            d.append(np.sqrt(np.mean((m.predict(Xp) - y[te]) ** 2)) - base)
        imp[c].append(np.mean(d))
tab_imp = pd.DataFrame({'Feature': cols4,
                        'dRMSE': [np.mean(imp[c]) for c in cols4],
                        'CI95': [ci95(imp[c]) for c in cols4]}
                       ).sort_values('dRMSE', ascending=False).round(4)
print(tab_imp.to_string(index=False))
print('\nNote: negative values mean that destroying the learned relationship IMPROVES '
      'prediction on unseen studies.')

tab_shap = None
if RUN_SHAP:
    try:
        import shap
        cols3 = FEATURE_SETS['F3 PTF + physics-guided']
        Xi, _ = impute_fold(get_X(cols3), get_X(cols3))
        m = clone(ET_FIXED).fit(Xi, y)
        sv = shap.TreeExplainer(m.named_steps['m']).shap_values(
            Xi[np.random.RandomState(SEED).choice(len(Xi), min(600, len(Xi)),
                                                  replace=False)],
            check_additivity=False)
        tab_shap = pd.DataFrame({'Feature': cols3,
                                 'mean_abs_SHAP': np.abs(sv).mean(0)}
                                ).sort_values('mean_abs_SHAP', ascending=False).round(4)
        print('\nTraining-set SHAP (for contrast only, NOT evidence of generalisation):\n'
              + tab_shap.to_string(index=False))
    except Exception as e:
        print('SHAP skipped:', e)


# =============================================================================
# PHASE 10 - DEPLOYABLE FOUR-INPUT MODEL + CONFORMAL INTERVALS
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 10 | DEPLOYABLE MODEL
#
# The field tool asks the user for clay, sand, organic carbon and bulk density, so
# the model that is distributed is evaluated with exactly those four inputs and
# under the grouped protocol that matches its intended use: a new site, not one
# already represented in the training data.
#
# Split-conformal calibration supplies the prediction interval. Within each outer
# training partition a quarter of the source datasets is withheld, the model is
# fitted on the remainder, and the 90th percentile of the absolute calibration
# residual becomes the interval half-width. Empirical coverage is then measured on
# the held-out fold. Calibrating on withheld studies rather than on random rows is
# what makes the interval meaningful for a new site, and comparing nominal with
# empirical coverage shows whether the exchangeability assumption holds.
#
# The applicability domain flags soils that lie far from the training data, using
# the mean distance to the five nearest training soils in the standardised input
# space, with the 95th percentile as the threshold.
#
# The serialised artefact contains the imputer, the model, the scaler, the
# conformal constant, the applicability threshold and the measured performance, so
# that the deployed pipeline and the evaluated pipeline are the same object.
# -----------------------------------------------------------------------------

banner('PHASE 10 | DEPLOYABLE MODEL EVALUATED WITH ITS OWN FOUR INPUTS')

rows, oof = [], np.full(len(y), np.nan)
X5 = get_X(FEATURE_SETS['F5 Four-input field set'])
qs = []
for k, (tr, te) in enumerate(sp_g):
    Xtr, Xte = impute_fold(X5[tr], X5[te])
    inner = list(GroupKFold(n_splits=N_INNER).split(Xtr, y[tr], groups_ds[tr]))
    s = RandomizedSearchCV(mdl, grid, n_iter=N_ITER, cv=inner, random_state=SEED,
                           scoring='neg_root_mean_squared_error', n_jobs=-1).fit(Xtr, y[tr])
    best = s.best_estimator_
    g = groups_ds[tr]; ug = pd.unique(g)
    rs = np.random.RandomState(SEED + k); rs.shuffle(ug)
    cal = np.isin(g, ug[:max(1, int(0.25 * len(ug)))])
    cm = clone(best).fit(Xtr[~cal], y[tr][~cal])
    q = float(np.quantile(np.abs(y[tr][cal] - cm.predict(Xtr[cal])), 0.90)); qs.append(q)
    yp = best.predict(Xte); oof[te] = yp
    r = metrics(y[te], yp)
    r.update(fold=k, q=q, Coverage=100 * np.mean((y[te] >= yp - q) & (y[te] <= yp + q)))
    rows.append(r)
dep = pd.DataFrame(rows)
Q90 = float(np.median(qs))
print(dep.round(3).to_string(index=False))
print(f'\nFour-input model, dataset-grouped CV: R2={dep.R2.mean():+.3f}, '
      f'RMSE={dep.RMSE.mean():.3f} log10 units, typical error a factor of '
      f'{dep.MedFactor.mean():.1f}, {dep.Within10.mean():.0f} % within one order.')
print(f'90 % conformal interval: half-width {Q90:.3f} log10 units '
      f'(spans a factor of {10**(2*Q90):.0f}); empirical coverage '
      f'{dep.Coverage.mean():.0f} %.')

# applicability domain
Xi5, _ = impute_fold(X5, X5)
sc = StandardScaler().fit(Xi5)
d5, _ = NearestNeighbors(n_neighbors=6).fit(sc.transform(Xi5)).kneighbors(sc.transform(Xi5))
dmean = d5[:, 1:].mean(1); AD_THR = float(np.quantile(dmean, 0.95))
ins = dmean <= AD_THR
tab_ad = pd.DataFrame({'Metric': ['AD threshold (mean distance to 5 nearest soils)',
                                  'Grouped-CV RMSE inside AD',
                                  'Grouped-CV RMSE outside AD'],
                       'Value': [round(AD_THR, 3),
                                 round(float(np.sqrt(np.mean((oof[ins]-y[ins])**2))), 3),
                                 round(float(np.sqrt(np.mean((oof[~ins]-y[~ins])**2))), 3)]})
print('\n' + tab_ad.to_string(index=False))

# final artefact
imp_full = IterativeImputer(estimator=BayesianRidge(), max_iter=10,
                            random_state=SEED).fit(X5)
final = ExtraTreesRegressor(n_estimators=NTREE, max_features=0.5, min_samples_leaf=2,
                            random_state=SEED, n_jobs=-1).fit(imp_full.transform(X5), y)
joblib.dump({'imputer': imp_full, 'model': final, 'scaler': sc,
             'features': FEATURE_SETS['F5 Four-input field set'],
             'training_X': X5, 'conformal_q90': Q90, 'nominal_level': 0.90,
             'ad_threshold': AD_THR, 'seed': SEED, 'target': 'log10 Ksat (cm/h)',
             'grouped_cv_r2': float(dep.R2.mean()),
             'grouped_cv_rmse': float(dep.RMSE.mean()),
             'median_multiplicative_error': float(dep.MedFactor.mean()),
             'empirical_coverage_grouped_cv': float(dep.Coverage.mean()),
             'note': 'Screening model. Order-of-magnitude estimates only; always report '
                     'the interval and the applicability-domain flag.'},
            f'{OUTDIR}/ksat_screening_model.joblib')
print(f'\nartefact written: {OUTDIR}/ksat_screening_model.joblib')


# =============================================================================
# PHASE 11 - FIGURES
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 11 | FIGURES
#
# Eight figures, all generated from the result tables produced above rather than
# from any separate calculation, so that no figure can disagree with the table it
# illustrates.
#
#   Fig 1  the sample: geography, target distribution, largest contributors
#   Fig 2  performance under the four validation protocols
#   Fig 3  feature-set ablation against the non-learned baselines
#   Fig 4  each algorithm under both protocols, the gap showing the optimism
#   Fig 5  predicted against observed under row-wise and grouped validation
#   Fig 6  out-of-fold permutation importance
#   Fig 7  leave-one-instrument-out and leave-one-country-out errors
#   Fig 8  the screening model with its prediction intervals and error
#          distribution
# -----------------------------------------------------------------------------

banner('PHASE 11 | FIGURES')

def savefig(fig, name):
    fig.savefig(f'{OUTDIR}/figures/{name}.png'); plt.close(fig); print('  ', name)

# Fig 1 - sample
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
d = df.dropna(subset=['lat', 'lon'])
c = d.groupby(['lat', 'lon']).size().reset_index(name='n')
ax[0].scatter(c['lon'], c['lat'], s=6 + 2.2 * np.sqrt(c['n']), c=CLR['a'], alpha=.65)
ax[0].set_xlim(-180, 180); ax[0].set_ylim(-60, 80)
ax[0].set_title('(a) Source datasets', loc='left'); ax[0].set_xlabel('Longitude')
ax[1].hist(df['logKsat'], bins=45, color=CLR['a'])
ax[1].set_title('(b) Target distribution', loc='left')
ax[1].set_xlabel('log10 Ksat (cm/h)')
top = df['Country'].value_counts().head(10)[::-1]
ax[2].barh(range(len(top)), top.values, color=CLR['c'])
ax[2].set_yticks(range(len(top))); ax[2].set_yticklabels(top.index, fontsize=7)
ax[2].set_title('(c) Largest contributors', loc='left')
savefig(fig, 'Fig1_sample')

# Fig 2 - protocols
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
x = np.arange(len(PROTOCOLS)); w = .26
for i, fs in enumerate(FS_MAIN):
    s = tab_protocols[tab_protocols.Features == fs].set_index('Protocol')
    s = s.loc[[p[0] for p in PROTOCOLS]]
    ax[0].bar(x + (i - 1) * w, s['R2'], w, yerr=s['R2ci'], capsize=2,
              color=[CLR['a'], CLR['b'], CLR['d']][i], label=fs)
    ax[1].bar(x + (i - 1) * w, s['RMSE'], w, yerr=s['RMSEci'], capsize=2,
              color=[CLR['a'], CLR['b'], CLR['d']][i])
for a, t in [(ax[0], 'Test R2'), (ax[1], 'RMSE (log10)')]:
    a.set_xticks(x); a.set_xticklabels([p[0].split(' ', 1)[1] for p in PROTOCOLS],
                                       fontsize=7.5, rotation=12)
    a.set_ylabel(t)
ax[0].axhline(0, color='k', lw=.8); ax[1].axhline(SD_Y, color='k', ls='--', lw=.9)
ax[0].legend(frameon=False, fontsize=7, loc='lower left')
savefig(fig, 'Fig2_protocols')

# Fig 3 - ablation
a3 = tab_abl.sort_values('RMSE')
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.barh(range(len(a3)), a3['RMSE'],
        color=[CLR['b'] if s[0] == 'B' else CLR['a'] for s in a3['FeatureSet']])
ax.set_yticks(range(len(a3))); ax.set_yticklabels(a3['FeatureSet'], fontsize=7.5)
ax.invert_yaxis(); ax.axvline(SD_Y, color='k', ls='--', lw=.9)
ax.set_xlabel('RMSE under dataset-grouped CV (log10)')
savefig(fig, 'Fig3_ablation')

# Fig 4 - algorithms
b = tab_alg.sort_values('R2_random'); yy = np.arange(len(b))
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.hlines(yy, b['R2_grouped'], b['R2_random'], color=CLR['g'])
ax.scatter(b['R2_random'], yy, s=32, color=CLR['b'], label='Random row-wise')
ax.scatter(b['R2_grouped'], yy, s=32, color=CLR['a'], label='Grouped by dataset')
ax.set_yticks(yy); ax.set_yticklabels(b['Algorithm'], fontsize=8)
ax.axvline(0, color='k', lw=.8); ax.set_xlabel('Test R2')
ax.legend(frameon=False, fontsize=8, loc='lower right')
savefig(fig, 'Fig4_algorithms')

# Fig 5 - predicted vs observed
fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.4)); lim = (-5.6, 4)
for a, key, t in [(ax[0], 'P1 Random row-wise|F3 PTF + physics-guided', '(a) Row-wise'),
                  (ax[1], 'P2 Grouped by dataset|F3 PTF + physics-guided', '(b) Grouped')]:
    yp = oof_store[key]; ok = ~np.isnan(yp)
    a.scatter(y[ok], yp[ok], s=5, alpha=.35, color=CLR['a'])
    a.plot(lim, lim, 'k-', lw=.9)
    a.plot(lim, [lim[0] + 1, lim[1] + 1], 'k--', lw=.6)
    a.plot(lim, [lim[0] - 1, lim[1] - 1], 'k--', lw=.6)
    a.set_xlim(lim); a.set_ylim(lim); a.set_title(t, loc='left')
    a.set_xlabel('Observed log10 Ksat')
ax[0].set_ylabel('Predicted log10 Ksat')
savefig(fig, 'Fig5_predicted_observed')

# Fig 6 - importance
i6 = tab_imp.head(14)[::-1]
fig, ax = plt.subplots(figsize=(5.6, 3.8))
ax.barh(range(len(i6)), i6['dRMSE'], xerr=i6['CI95'].fillna(0), capsize=2, color=CLR['c'])
ax.set_yticks(range(len(i6))); ax.set_yticklabels(i6['Feature'], fontsize=8)
ax.axvline(0, color='k', lw=.8); ax.set_xlabel('Increase in RMSE when permuted')
savefig(fig, 'Fig6_importance')

# Fig 7 - transfer
fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.4))
for a, t, title in [(ax[0], tab_inst, '(a) Leave-one-instrument-out'),
                    (ax[1], tab_ctry, '(b) Leave-one-country-out')]:
    t = t.sort_values('RMSE')
    a.barh(range(len(t)), t['RMSE'], color=CLR['a'])
    a.set_yticks(range(len(t))); a.set_yticklabels(t['Level'], fontsize=7.5)
    a.invert_yaxis(); a.axvline(SD_Y, color='k', ls='--', lw=.9)
    a.set_xlabel('RMSE (log10)'); a.set_title(title, loc='left')
savefig(fig, 'Fig7_transfer')

# Fig 8 - screening model
ok = ~np.isnan(oof); o = np.argsort(y[ok])
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.3))
ax[0].fill_between(np.arange(ok.sum()), (oof[ok] - Q90)[o], (oof[ok] + Q90)[o],
                   color=CLR['a'], alpha=.25, lw=0, label='90 % interval')
ax[0].plot(np.arange(ok.sum()), y[ok][o], 'k', lw=.9, label='Observed')
ax[0].set_xlabel('Records ranked by observed Ksat'); ax[0].set_ylabel('log10 Ksat')
ax[0].legend(frameon=False, fontsize=7.5, loc='upper left')
ax[1].hist(oof[ok] - y[ok], bins=45, color=CLR['b'])
ax[1].axvline(0, color='k', lw=.9); ax[1].set_xlabel('Prediction error (log10)')
savefig(fig, 'Fig8_screening_model')


# =============================================================================
# PHASE 12 - EXPORT EVERY TABLE
# =============================================================================
# -----------------------------------------------------------------------------
# PHASE 12 | EXPORT
#
# Everything needed to write up, check or reproduce the analysis:
#
#   results_tables.xlsx           every table, one sheet each
#   analysis_dataset.csv          the analysis sample with its grouping labels and
#                                 constructed features
#   ksat_screening_model.joblib   the deployable artefact
#   run_metadata.json             seed, fold counts, search budget, sample sizes
#                                 and library versions
#   figures/                      the eight figures at publication resolution
# -----------------------------------------------------------------------------

banner('PHASE 12 | EXPORT')

sheets = {'S0_sample_flow': tab_flow, 'S1_missingness': tab_missing,
          'S2_structure': tab_groups, 'S3_protocols': tab_protocols,
          'S3b_protocol_tests': tab_protocol_test,
          'S3c_duplicate_effect': tab_duplicates,
          'S4_decomposition': tab_decomp, 'S5_ablation': tab_abl,
          'S6_algorithms': tab_alg, 'S7_variance': tab_var,
          'S8_leave_instrument': tab_inst, 'S9_leave_country': tab_ctry,
          'S10_perm_importance': tab_imp, 'S11_deployment': dep.round(3),
          'S12_applicability': tab_ad}
if tab_shap is not None:
    sheets['S13_shap'] = tab_shap

with pd.ExcelWriter(f'{OUTDIR}/results_tables.xlsx', engine='openpyxl') as w:
    for nm, t in sheets.items():
        t.to_excel(w, sheet_name=nm[:31], index=False)

keep = ['Code', 'DatasetID', 'Provider', 'Country', 'lat', 'lon', 'GeoBlock',
        'Clay', 'Silt', 'Sand', 'OC', 'Db', 'dg', 'Sg', 'Instrument',
        'Landuse_Code', 'Texture Class', 'Ksat', 'logKsat'] + PHYS
df[[c for c in keep if c in df.columns]].to_csv(f'{OUTDIR}/analysis_dataset.csv',
                                                index=False)

env = {'seed': SEED, 'n_outer': N_OUTER, 'n_inner': N_INNER, 'n_iter': N_ITER,
       'n_repeats': N_REPEATS, 'drop_exact_duplicates': DROP_EXACT_DUPLICATES,
       'n_duplicates_removed': int(n_dup),
       'fast_mode': FAST_MODE, 'n_records': int(len(df)),
       'n_datasets': int(df['DatasetID'].nunique()),
       'python': sys.version.split()[0], 'numpy': np.__version__,
       'pandas': pd.__version__}
import sklearn
env['scikit_learn'] = sklearn.__version__
json.dump(env, open(f'{OUTDIR}/run_metadata.json', 'w'), indent=2)

print(f'  {OUTDIR}/results_tables.xlsx')
print(f'  {OUTDIR}/analysis_dataset.csv')
print(f'  {OUTDIR}/ksat_screening_model.joblib')
print(f'  {OUTDIR}/run_metadata.json')
print(f'  {OUTDIR}/figures/  (8 figures)')

banner('HEADLINE NUMBERS')
p1 = tab_protocols[(tab_protocols.Protocol == 'P1 Random row-wise') &
                   (tab_protocols.Features == 'F4 F3 + measurement metadata')].iloc[0]
p2 = tab_protocols[(tab_protocols.Protocol == 'P2 Grouped by dataset') &
                   (tab_protocols.Features == 'F4 F3 + measurement metadata')].iloc[0]
print(f'Same pipeline, same features, only the fold-assignment rule differs:')
print(f'  random row-wise split : R2 = {p1.R2:+.3f}   RMSE = {p1.RMSE:.3f}')
print(f'  grouped by dataset    : R2 = {p2.R2:+.3f}   RMSE = {p2.RMSE:.3f}')
print(f'  target standard dev.  :               {SD_Y:.3f}')
print(f'  mean optimism across {len(tab_alg)} algorithms: {tab_alg.Optimism.mean():+.3f} R2')
sk1 = tab_protocols[(tab_protocols.Protocol == 'P1 Random row-wise') &
                    (tab_protocols.Features == 'F4 F3 + measurement metadata')].iloc[0]
sk2 = tab_protocols[(tab_protocols.Protocol == 'P2 Grouped by dataset') &
                    (tab_protocols.Features == 'F4 F3 + measurement metadata')].iloc[0]
print(f'  skill against the training mean: {sk1.Skill:+.3f} row-wise, '
      f'{sk2.Skill:+.3f} grouped')
print('\nDone.')

try:
    from google.colab import files as _f
    import shutil
    shutil.make_archive(OUTDIR, 'zip', OUTDIR)
    _f.download(f'{OUTDIR}.zip')
except Exception:
    pass


PHASE 1 | DATA INGESTION
Select SWIG_database.xlsx from your computer:


Saving SWIG database.xlsx to SWIG database.xlsx
Using: SWIG database.xlsx
Sheets found: Read me, T_Hour, I_cm, Tension_cm, Metadata, Locations, Statistics, Ref. for digitized data, Ref. for data provided by owner
Metadata sheet "Metadata": 5023 rows, 38 columns
Locations sheet "Locations": 208 contributed blocks

PHASE 2 | SAMPLE DEFINITION (reported as a flow table)
                                                  Step  Records remaining
                          Infiltration records in SWIG               5023
                    Records with a reported Ksat value               1895
                          Ksat > 0 (log-transformable)               1893
             Excluded: bulk density > 2.65 g/cm3 (n=1)               1892
    Excluded: texture not summing to 100 +/- 5 % (n=0)               1892
        Excluded: no core soil property reported (n=9)               1883
Excluded: exact duplicate of a retained record (n=825)               1058
                                    FI

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>